# Day 56 · Exercise 1: File Upload Endpoint

**What you'll build:** Implement `build_upload_api()` — a FastAPI app with `POST /upload` that accepts a file via multipart form and returns its metadata. This is the entry point to every file-upload system.

## Setup (provided)

In [ ]:
from fastapi import FastAPI, UploadFile, File
from starlette.testclient import TestClient


## Your Implementation

In [ ]:
def build_upload_api() -> FastAPI:
    """Build a FastAPI app with a single file-upload endpoint.

    The endpoint POST /upload should:
      - Accept a single file via multipart form (UploadFile = File(...))
      - Read the file bytes with `content = await file.read()`
      - Return JSON: {"filename": ..., "content_type": ..., "size": len(content)}

    Returns:
        A FastAPI app instance.
    """
    app = FastAPI()

    # TODO: add an async def POST /upload that uses UploadFile = File(...)
    #       reads with await, returns {"filename", "content_type", "size"}

    return app


In [ ]:
def build_upload_api() -> FastAPI:
    app = FastAPI()

    @app.post("/upload")
    async def upload(file: UploadFile = File(...)):
        content = await file.read()
        return {
            "filename": file.filename,
            "content_type": file.content_type,
            "size": len(content),
        }

    return app


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        app = build_upload_api()
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: build_upload_api not implemented")
        print(f"\nScore: 0 / {total}")
        return
    except Exception as e:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: {type(e).__name__}: {e}")
        print(f"\nScore: 0 / {total}")
        return

    client = TestClient(app, raise_server_exceptions=False)

    # check 1: POST /upload returns 200
    r = client.post("/upload",
                    files={"file": ("hello.txt", b"hello world", "text/plain")})
    _chk(1, r.status_code == 200,
         f"POST /upload with text file → 200 (got {r.status_code})")

    if r.status_code == 200:
        data = r.json()
        _chk(2, data.get("filename") == "hello.txt",
             f"filename == 'hello.txt' (got {data.get('filename')})")
        _chk(3, data.get("content_type") == "text/plain",
             f"content_type == 'text/plain' (got {data.get('content_type')})")
        _chk(4, data.get("size") == 11,
             f"size == 11 (got {data.get('size')})")
    else:
        for i in range(2, 5):
            print(f"  ❌ Check {i}: skipped (check 1 failed)")

    # check 5: upload a different file
    r2 = client.post("/upload",
                     files={"file": ("data.csv", b"a,b,c\n1,2,3", "text/csv")})
    _chk(5, r2.status_code == 200 and r2.json().get("size") == 11,
         f"second upload returns correct size (got status={r2.status_code})")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Add a second endpoint `POST /upload-multiple` that accepts `files: list[UploadFile] = File(...)` and returns a list of metadata dicts — one per file. Test with TestClient using multiple `files=[('file', ...), ('file', ...)]` entries in the same request.

## Solution

<details>
<summary>Show solution</summary>

```python
def build_upload_api() -> FastAPI:
    app = FastAPI()

    @app.post("/upload")
    async def upload(file: UploadFile = File(...)):
        content = await file.read()
        return {
            "filename": file.filename,
            "content_type": file.content_type,
            "size": len(content),
        }

    return app
```

**Why this works:** `UploadFile = File(...)` tells FastAPI to parse a multipart
form field named `file`. The handler must be `async def` because `file.read()`
is a coroutine — it reads from the underlying spooled temporary file. TestClient
supports async route handlers transparently. The JSON response is built from
`file.filename`, `file.content_type`, and `len(content)`.

</details>